In [1]:
!pip install -q -U transformers accelerate bitsandbytes pymupdf qwen-vl-utils[decord]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 155.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 149.4 MB/s eta 0:00:00


In [2]:
import torchvision
import torchaudio
import torch
import fitz
import os
from transformers import AutoProcessor, AutoModel, AutoTokenizer, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from pathlib import Path
from qwen_vl_utils import process_vision_info

In [3]:
def pdf_to_images(pdf_path: str, out_dir: str, dpi: int = 300):
  os.makedirs(out_dir, exist_ok=True)
  doc = fitz.open(pdf_path)
  zoom = dpi / 72
  mat = fitz.Matrix(zoom, zoom)

  image_paths = []
  for i, page in enumerate(doc):
    if i >30:                                        # test breaking
      break
    pix = page.get_pixmap(matrix=mat)
    img_path = os.path.join(out_dir, f"page_{i+1:03d}.png")
    pix.save(img_path)
    image_paths.append(img_path)
    print(f"Rendered {img_path}")

  doc.close()
  return image_paths


def run_ocr(image_paths, model_id: str = "Qwen/Qwen3-VL-8B-Instruct"):
  """
  - 4-bit quantization
  - load in model
  - run inference on  on all images
  """
  bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_compute_dtype=torch.bfloat16,
  bnb_4bit_quant_type="nf4",
)

  processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256*28*28,
    max_pixels=1024*28*28,  # lower this further (e.g. 768*28*28) if still tight
)
  model = Qwen3VLForConditionalGeneration.from_pretrained(
      model_id,
      trust_remote_code=True,
      device_map={"": 0},
      torch_dtype=torch.bfloat16,
      quantization_config=bnb_config,
      low_cpu_mem_usage=True,
  ).eval()

  results = {}
  for img_path in image_paths:
      messages = [
      {
          "role": "user",
          "content": [
              {"type": "image", "image": f"{img_path}"},
              {"type": "text", "text":
                """Look at this image and determine if it contains an actual table — that is, structured content with visible rows and columns, gridlines or clear column alignment, and multiple data fields per row.

                  Do NOT treat the following as tables:
                  - Body text or prose paragraphs, even if organized under headings
                  - Single-column lists of headings followed by descriptive text
                  - Definition-style content (a term followed by an explanatory paragraph)

                  If NO real table is present, respond with exactly:
                  {"table_found": false}

                  If a real table IS present, extract it with this format:
                  {
                    "table_found": true,
                    "table_name": "<inferred from caption/context>",
                    "rows": [
                      ["header1", "header2", ...],
                      ["row1val1", "row1val2", ...]
                    ]
                  }

                  Rules:
                  - Output valid JSON only, no explanation before or after
                  - Preserve merged cells by repeating the value across merged rows/columns
                  - If a cell is empty or unreadable, use null
                  - Keep numbers as strings if they include currency symbols, %, or commas
                  """}
              ]
          }
      ]

      text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
      image_inputs, video_inputs = process_vision_info(messages)
      inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

      out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
      trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
      results[img_path] = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
      print(f"OCR done: {img_path}")

  return results

In [4]:
pdf_path = "filing.pdf"
out_dir = "pages"

images = pdf_to_images(pdf_path, out_dir, dpi=300)

MuPDF error: format error: non-page object in page tree

Rendered pages/page_001.png
Rendered pages/page_002.png
Rendered pages/page_003.png
Rendered pages/page_004.png
Rendered pages/page_005.png
Rendered pages/page_006.png
Rendered pages/page_007.png
Rendered pages/page_008.png
Rendered pages/page_009.png
Rendered pages/page_010.png
Rendered pages/page_011.png
Rendered pages/page_012.png
Rendered pages/page_013.png
Rendered pages/page_014.png
Rendered pages/page_015.png
Rendered pages/page_016.png
Rendered pages/page_017.png
Rendered pages/page_018.png
Rendered pages/page_019.png
Rendered pages/page_020.png
Rendered pages/page_021.png
Rendered pages/page_022.png
Rendered pages/page_023.png
Rendered pages/page_024.png
Rendered pages/page_025.png
Rendered pages/page_026.png
Rendered pages/page_027.png
Rendered pages/page_028.png
Rendered pages/page_029.png
Rendered pages/page_030.png
Rendered pages/page_031.png


In [5]:
ocr_results = run_ocr(images)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


OCR done: pages/page_001.png
OCR done: pages/page_002.png
OCR done: pages/page_003.png
OCR done: pages/page_004.png
OCR done: pages/page_005.png
OCR done: pages/page_006.png
OCR done: pages/page_007.png
OCR done: pages/page_008.png
OCR done: pages/page_009.png
OCR done: pages/page_010.png
OCR done: pages/page_011.png
OCR done: pages/page_012.png
OCR done: pages/page_013.png
OCR done: pages/page_014.png
OCR done: pages/page_015.png
OCR done: pages/page_016.png
OCR done: pages/page_017.png
OCR done: pages/page_018.png
OCR done: pages/page_019.png
OCR done: pages/page_020.png
OCR done: pages/page_021.png
OCR done: pages/page_022.png
OCR done: pages/page_023.png
OCR done: pages/page_024.png
OCR done: pages/page_025.png
OCR done: pages/page_026.png
OCR done: pages/page_027.png
OCR done: pages/page_028.png
OCR done: pages/page_029.png
OCR done: pages/page_030.png
OCR done: pages/page_031.png


In [6]:
ocr_results

{'pages/page_001.png': '{"table_found": false}',
 'pages/page_002.png': '{\n  "table_found": true,\n  "table_name": "Contents",\n  "rows": [\n    ["Strategic report", "1"],\n    ["Directors\' report", "12"],\n    ["Current directors", "16"],\n    ["Risk management", "17"],\n    ["Forward looking statements", "64"],\n    ["Independent auditors\' report", "65"],\n    ["Consolidated income statement", "76"],\n    ["Consolidated statement of comprehensive income", "77"],\n    ["Consolidated balance sheet", "78"],\n    ["Consolidated statement of changes in equity", "79"],\n    ["Consolidated cash flow statement", "82"],\n    ["Notes to the consolidated financial statements", "83"],\n    ["Bank balance sheet", "164"],\n    ["Bank statement of changes in equity", "165"],\n    ["Bank cash flow statement", "167"],\n    ["Notes to the Bank financial statements", "168"],\n    ["Subsidiaries and related undertakings", "202"]\n  ]\n}',
 'pages/page_003.png': '{"table_found": false}',
 'pages/page_

In [7]:
ocr_results

{'pages/page_001.png': '{"table_found": false}',
 'pages/page_002.png': '{\n  "table_found": true,\n  "table_name": "Contents",\n  "rows": [\n    ["Strategic report", "1"],\n    ["Directors\' report", "12"],\n    ["Current directors", "16"],\n    ["Risk management", "17"],\n    ["Forward looking statements", "64"],\n    ["Independent auditors\' report", "65"],\n    ["Consolidated income statement", "76"],\n    ["Consolidated statement of comprehensive income", "77"],\n    ["Consolidated balance sheet", "78"],\n    ["Consolidated statement of changes in equity", "79"],\n    ["Consolidated cash flow statement", "82"],\n    ["Notes to the consolidated financial statements", "83"],\n    ["Bank balance sheet", "164"],\n    ["Bank statement of changes in equity", "165"],\n    ["Bank cash flow statement", "167"],\n    ["Notes to the Bank financial statements", "168"],\n    ["Subsidiaries and related undertakings", "202"]\n  ]\n}',
 'pages/page_003.png': '{"table_found": false}',
 'pages/page_